# J1939 Digital Annex to JSON
This notebook is designed to step through the process of converting the SAE J1939 Digital Annex to a json file. The resulting json file can be loaded into an application to interpret the controller area network (CAN) data coming from a J1939 network. 

### Intellectual Property
This notebook is released under an MIT License. However, it operates on a copyrighted data source owned by SAE International. Therefore, this repository does not distribute either the J1939 database or the resulting JSON file. To use this notebook, you must have a licensed copy of the Digital Annex available on your local computer. 

SAE J1939 Digial Annex is available to purchase at https://www.sae.org/standards/content/j1939da_202301/. This link may point to a historical document. 

If your institution has a subscription to the SAE Mobilus, then you may be able to download the document based on your subscription. 

### Approach
This notebook utilizes the pandas library for Python.

https://pandas.pydata.org/

The pandas library has some well functioning importers for Excel, so it can read in the SAE J1939 Digital Annex easily. However, some updated libraries may need to be installed. The installation for these supporting importers are commented out in the notebook. 

The notebook can be run cell by cell in a psuedo-interactive form so you can see the different features being extracted from the database. For a quick result, simply select `Restart and Run All` from the Kernel menu.

### Limitations
The notebook may not work on all versions of the SAE J1939 Digial Annex. From time to time, the authors of the DA will change the headings or formats. 

Also, some useful data, like the interpretation of switch status data, is not available in the digital annex. These data may have to be manually input from the J1939-71 document.

In [ ]:
import json
import pandas

In [ ]:
#!pip install --upgrade --user numexpr

In [ ]:
#!pip install  --upgrade --user openpyxl

## Load the SLOTs

In [ ]:
with open('J1939DA OCT2021.xlsx', 'rb') as io:
    slots_df = pandas.read_excel(io, sheet_name='SLOTs',
                                 usecols="B:P", 
                                 engine='openpyxl',
                                 index_col=0,
                                 header = 3)
slots_df

In [ ]:
reduced_slots = slots_df[["Length",
                          "Unit",
                          "Scale Factor\n(value only)",
                          "Offset\n(value only)",
                          "Range Maximum\n(value only)",
                          "Length Minimum\n(bits)",
                          "Length Maximum\n(bits)"]].copy()
reduced_slots.rename(columns={"Scale Factor\n(value only)": 'Resolution', 
                             "Offset\n(value only)": 'Offset',
                             "Range Maximum\n(value only)":'OperationalHigh',
                             'Length Minimum\n(bits)':"Length",
                             "Length":"byte length",
                             "Length Maximum\n(bits)":"Max Bits"},
                    inplace=True)

reduced_slots.dropna(how='any',subset=["Length","Max Bits"],inplace=True)

reduced_slots[["Length","Max Bits"]] = reduced_slots[["Length","Max Bits"]].astype('int')
reduced_slots

In [ ]:
reduced_slots.groupby('Length').groups

In [ ]:
slot_dict = reduced_slots.to_dict('index')
slot_dict

In [ ]:
#Create a pandas data frame with the SPN as the index column and all the data from the SPs and PGs table.
with open('J1939DA OCT2021.xlsx', 'rb') as io:
    spn_df = pandas.read_excel(io, sheet_name='SPs & PGs',
                                 usecols="E:AP", 
                                 header = 3,
                                 converters={'SPN':int,
                                             'PGN':int,
                                             'EDP':int,
                                             'DP':int,
                                             'PF':int,
                                             'SLOT Identifier':int,
                                             'Length Minimum\n(bits)':int,
                                             'Length Maximum\n(bits)':int})
spn_df

### Parameter Group Numbers

In [ ]:
pgns = spn_df.groupby(by='PGN')
pgns.groups

In [ ]:
reduced_pgns = (spn_df.drop_duplicates('PGN')[['PGN','PG Label','Transmission Rate','PG Acronym','PG Data Length']])
reduced_pgns.dropna(how='any',subset=["PGN"],inplace=True)
reduced_pgns.set_index('PGN',inplace=True)
reduced_pgns.rename(columns={'PG Label': 'Name', 
                             'PG Acronym': 'Label',
                             'PG Data Length':'PGNLength',
                             'Transmission Rate':"Rate"},
                    inplace=True)
reduced_pgns

In [ ]:
pgn_dict = reduced_pgns.to_dict(orient='index')
for pgn_key in reduced_pgns.index:
    pgn = int(pgn_key)
    pgn_dict[pgn]["SPNs"] = sorted(spn_df["SPN"][pgns.groups[pgn]].dropna().to_list())
pgn_dict

In [ ]:
pgn_dict[65265]

In [ ]:
pgn_dict[0]

In [ ]:
try:
    pgn_dict[1]
except KeyError:
    print("Key Error - Entry does not exist.")

## Suspect Parameter Numbers

In [ ]:
#%pip install --upgrade --user pandas

In [ ]:
reduced_spns = spn_df.drop_duplicates('SPN')[['SPN',
                                              'PGN',
                                              'SP Label',
                                              'Transmission Rate',
                                              'PG Acronym',
                                              'Length Minimum\n(bits)',
                                              'SLOT Identifier',
                                              'SP Position in PG',
                                              'SP Start Bit'
                                             ]]
reduced_spns.dropna(how='any',subset=["SPN"],inplace=True)
reduced_spns.set_index('SPN',inplace=True)
reduced_spns.rename(columns={'SP Label': 'Name', 
                             'PG Acronym': 'Acronym',
                             'PG Data Length':'PGNLength',
                             'Length Minimum\n(bits)':'SPNLength',
                             'Transmission Rate':"TransmissionRate"},
                    inplace=True)
reduced_spns

In [ ]:
def get_bit_loc(position):
    bite_bit = position.split(".")
    try:
        bite = int(bite_bit[0])
        bit  = int(bite_bit[1])
        return int((bite-1)*8 + (bit-1))
    except ValueError:
        return -1
    
reduced_spns['StartBit'] = reduced_spns.apply(
    lambda x: get_bit_loc(str(x[['SP Start Bit']].values[0])),
    axis = 1)

reduced_spns['StartBit'] = reduced_spns['StartBit'].astype(int)
reduced_spns

In [ ]:
spn_dict = reduced_spns.to_dict(orient='index')
spn_dict

In [ ]:
for k,v in spn_dict.items():
    slot = v["SLOT Identifier"]
    try:
        spn_dict[k].update(slot_dict[slot])
    except KeyError:
        pass
spn_dict

In [ ]:
spn_dict[190]

In [ ]:
spn_dict[84]

In [ ]:
spn_dict[237]

### US Units

In [ ]:
us_slot_dict = slot_dict.copy()
for k,v in slot_dict.items():
    if v["Unit"] == 'km':
        factor = 0.6213712
        us_slot_dict[k]["Unit"] = 'miles'
        us_slot_dict[k]["Resolution"] = v["Resolution"]*factor
        us_slot_dict[k]["Offset"] = v["Offset"]*factor
        us_slot_dict[k]["OperationalHigh"] = v["OperationalHigh"]*factor
    elif v["Unit"] == 'km/h':
        factor = 0.6213712
        us_slot_dict[k]["Unit"] = 'mph'
        us_slot_dict[k]["Resolution"] = v["Resolution"]*factor
        us_slot_dict[k]["Offset"] = v["Offset"]*factor
        us_slot_dict[k]["OperationalHigh"] = v["OperationalHigh"]*factor
    elif v["Unit"] == 'kpa':
        factor = 0.1450377
        us_slot_dict[k]["Unit"] = 'psi'
        us_slot_dict[k]["Resolution"] = v["Resolution"]*factor
        us_slot_dict[k]["Offset"] = v["Offset"]*factor
        us_slot_dict[k]["OperationalHigh"] = v["OperationalHigh"]*factor
    elif v["Unit"] == 'm':
        factor = 3.28084
        us_slot_dict[k]["Unit"] = 'ft'
        us_slot_dict[k]["Resolution"] = v["Resolution"]*factor
        us_slot_dict[k]["Offset"] = v["Offset"]*factor
        us_slot_dict[k]["OperationalHigh"] = v["OperationalHigh"]*factor
us_slot_dict[74]

In [ ]:
#US Version
for k,v in spn_dict.items():
    slot = v["SLOT Identifier"]
    try:
        spn_dict[k].update(us_slot_dict[slot])
    except KeyError:
        pass
spn_dict[84]

## Source Addresses

In [ ]:
#Create a pandas data frame with the Source Addresses
with open('J1939DA OCT2021.xlsx', 'rb') as io:
    global_sa_df = pandas.read_excel(io, sheet_name='Global Source Addresses (B2)',
                                 usecols="B:D", 
                                 index_col=0,
                                 header = 3)
global_sa_df

In [ ]:
global_sa_df.loc[[128]]

In [ ]:
global_sa_df.drop(index=128,inplace=True)

In [ ]:
#Create a pandas data frame with the Source Addresses
with open('J1939DA OCT2021.xlsx', 'rb') as io:
    ig1_sa_df = pandas.read_excel(io, sheet_name='IG1 Source Addresses (B3)',
                                 usecols="B:D", 
                                 index_col=0,
                                 header = 3)
ig1_sa_df

In [ ]:
ig1_sa_df.loc[[128]]

In [ ]:
ig1_sa_df.drop(index=128,inplace=True)

In [ ]:
sa_df = global_sa_df.merge(ig1_sa_df,how='outer',
                           on=["Source Address ID","Name","Notes"], 
                           sort=True)
sa_df

In [ ]:
sa_dict = sa_df[['Name']].to_dict('index')
sa_dict

In [ ]:
sa_df.drop(index=208, inplace=True)

In [ ]:
sa_dict={}
for k in sa_df.index.values:
    sa_dict[int(k)]=sa_df[["Name"]].loc[k].values[0]
sa_dict

### Failure Mode Indicators

Note:  FMIs 22-30 are Reserved For SAE Assignment

In [ ]:
fmi_dict = {
    0: {"Name": "Data Valid But Above Normal Operational Range",
        "Severity": "Most Severe Level"},
    1: {"Name": "Data Valid But Below Normal Operational Range",
        "Severity": "Most Severe Level"},
    2: {"Name": "Data Erratic, Intermittent Or Incorrect",
        "Severity": "N/A"},
    31: {"Name": "Condition Exists",
         "Severity": "N/A"}
}

## Create the JSON files

In [ ]:
combined_dict = {
    "J1939SATabledb":sa_dict,
    "J1939SPNdb": spn_dict,
    "J1939PGNdb": pgn_dict,
    "J1939FMITabledb": fmi_dict
}
with open('J1939db.json','w') as f:
    json.dump(combined_dict,f,sort_keys=True)

### Cleanup

In [ ]:
with open('J1939db.json','r') as f:
    j1939db = json.load(f)

In [ ]:
#This entry has a strange character and new line
j1939db["J1939PGNdb"]["0"]["Rate"]

In [ ]:
#This entry has a strange apostrophe
j1939db["J1939PGNdb"]["20736"]["Rate"]

In [ ]:
with open('J1939db.json','r') as f:
    json_text = f.read()

json_text = json_text.replace(r'_x000D_','')
json_text = json_text.replace(r'\n','; ')
json_text = json_text.replace(r"\u2019","'")
with open('J1939db.json','w') as f:
    f.write(json_text)

In [ ]:
with open('J1939db.json','r') as f:
    j1939db = json.load(f)
#Show the newline is replaced with a semicolon    
j1939db["J1939PGNdb"]["0"]["Rate"]

In [ ]:
#Show the apostrophe is in place.
j1939db["J1939PGNdb"]["20736"]["Rate"]